<a href="https://colab.research.google.com/github/AnjanPayra/3-MEM/blob/main/3_MEM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
essential_protein_prediction.py
=================================
End-to-end, self-contained pipeline for predicting essential proteins in the
YDIP, YMIPS, YHQ and YMBD yeast PPI networks, validated against a curated
essential / non-essential gold-standard gene set (Essential.xlsx).

USAGE
-----
    python essential_protein_prediction.py

Expects the following files in UPLOAD_DIR (edit the constant below if your
paths differ):
    YDIP.txt, YMIPS.txt, YHQ.txt, YMBD.txt          -- PPI edge lists
    YDIP_GO_final.csv, YHQ_GO_final.csv,
    YMBD_GO_final.csv, YMIPS_GO_final.csv           -- per-protein GO score
    Physico_chemical.csv                            -- 11 physico-chemical descriptors
    YDIP_level1_COG.csv                             -- COG/NOG/KOG orthology mapping (YDIP only)
    Essential.xlsx (sheets "essential proteins" /
                     "non-essential proteins")      -- gold standard labels

METHOD SUMMARY
--------------
1. TOPOLOGY   -- nine classic PPI centrality/essentiality measures per
   network: Degree (DC), Betweenness (BC, pivot-approximated for speed),
   Closeness (CC), Eigenvector (EC), Subgraph centrality (SC, approximated
   from the top-100 adjacency eigenpairs), Clustering coefficient (CL),
   Local Average Connectivity (LAC), Neighbourhood Centrality
   (NC = own degree + neighbours' degree), and k-core number (KC).
2. BIOLOGY    -- a precomputed GO semantic-similarity score (GO); 11
   physico-chemical sequence descriptors (pc_1..pc_11); for YDIP an
   orthology-conservation feature (ORTHO = number of distinct COG/NOG/KOG
   groups a protein belongs to).
3. VALIDATION -- (a) top-N% ranking precision/recall/F1 for each individual
   score (the classic "centrality-lethality rule" check), and (b) a Random
   Forest classifier trained on all features jointly, evaluated with
   stratified 10-fold cross-validation (Accuracy, Sensitivity, Specificity,
   PPV, NPV, F1, MCC, ROC-AUC, PR-AUC) using out-of-fold predictions.
"""
import csv
import json
import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.sparse.linalg import eigsh
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                              precision_recall_curve, matthews_corrcoef, confusion_matrix)


In [2]:
# ============================== data loading ==============================
UPLOAD_DIR = '/mnt/user-data/uploads'

def load_edge_list(path):
    """Parse a 2-column CSV (with header) of PPI edges, tolerant of \r
 and
    variable whitespace. Returns a list of (protein1, protein2) tuples."""
    edges = []
    with open(path) as f:
        lines = f.read().splitlines()
    for line in lines[1:]:
        parts = [p.strip() for p in line.split(',')]
        if len(parts) >= 2 and parts[0] and parts[1]:
            edges.append((parts[0], parts[1]))
    return edges

def build_graph(name):
    """Build an undirected, simple (no self-loop, no multi-edge) graph for a
    named PPI dataset (YDIP / YMIPS / YHQ / YMBD)."""
    edges = load_edge_list(f'{UPLOAD_DIR}/{name}.txt')
    G = nx.Graph()
    G.add_edges_from(edges)
    G.remove_edges_from(list(nx.selfloop_edges(G)))
    isolates = list(nx.isolates(G))
    G.remove_nodes_from(isolates)
    return G

def load_gold_standard():
    """Return dict protein -> 1 (essential) / 0 (non-essential) from the
    curated Essential.xlsx gold-standard sheet."""
    xl = pd.ExcelFile(f'{UPLOAD_DIR}/Essential.xlsx')
    ess = set(xl.parse('essential proteins').iloc[:, 0].astype(str).str.strip())
    non = set(xl.parse('non-essential proteins').iloc[:, 0].astype(str).str.strip())
    labels = {p: 1 for p in ess}
    labels.update({p: 0 for p in non})
    return labels

def load_go_scores(name):
    """protein -> GO semantic-similarity based score (precomputed)."""
    df = pd.read_csv(f'{UPLOAD_DIR}/{name}_GO_final.csv', header=None, names=['protein', 'go_score'])
    df['protein'] = df['protein'].astype(str).str.strip()
    return df.set_index('protein')['go_score'].to_dict()

def load_physico_chemical():
    """protein -> array of 11 physico-chemical descriptors."""
    df = pd.read_csv(f'{UPLOAD_DIR}/Physico_chemical.csv', header=None)
    df[0] = df[0].astype(str).str.strip()
    df = df.set_index(0)
    df.columns = [f'pc_{i + 1}' for i in range(df.shape[1])]
    return df

def load_cog_orthology(name='YDIP'):
    """protein -> number of distinct COG/NOG/KOG orthologous-group
    annotations (only available for YDIP in the supplied data). Used as a
    conservation / orthology-based feature: more independently-verified
    orthologous groups a protein belongs to, the more evolutionarily
    conserved -- a property empirically associated with essentiality."""
    path = f'{UPLOAD_DIR}/{name}_level1_COG.csv'
    counts = {}
    with open(path) as f:
        reader = csv.reader(f)
        for row in reader:
            row = [c.strip() for c in row if c.strip()]
            if not row:
                continue
            protein, cogs = (row[0], row[1:])
            counts[protein] = len(cogs)
    return counts

In [3]:
# ============================== topology features ==============================
def _approx_subgraph_centrality(G: nx.Graph, nodes, k=100, seed=42):
    """Approximate subgraph centrality SC(i) = sum_j v_j(i)^2 * exp(lambda_j)
    using only the k largest-magnitude eigenpairs of the adjacency matrix
    (ARPACK, sparse) instead of a full dense matrix exponential. This keeps
    the O(n^3) dense eigendecomposition from becoming a bottleneck on
    networks of several thousand nodes while still capturing the dominant
    spectral contributions (subgraph centrality is dominated by the largest
    eigenvalues)."""
    A = nx.to_scipy_sparse_array(G, nodelist=nodes, format='csr', dtype=float)
    n = A.shape[0]
    k = min(k, n - 2) if n > 3 else 1
    try:
        eigvals, eigvecs = eigsh(A, k=k, which='LM')
    except Exception:
        eigvals, eigvecs = np.linalg.eigh(A.toarray())
    weights = np.exp(eigvals)
    sc_vals = eigvecs ** 2 @ weights
    return dict(zip(nodes, sc_vals))

def compute_all(G: nx.Graph) -> pd.DataFrame:
    nodes = list(G.nodes())
    deg = dict(G.degree())
    n = G.number_of_nodes()
    dc = {n_: deg[n_] for n_ in nodes}
    bc_k = min(n, 300) if n > 1000 else None
    bc = nx.betweenness_centrality(G, k=bc_k, normalized=True, seed=42)
    cc = nx.closeness_centrality(G)
    try:
        ec = nx.eigenvector_centrality(G, max_iter=2000, tol=1e-06)
    except nx.PowerIterationFailedConvergence:
        ec = nx.eigenvector_centrality_numpy(G)
    sc = _approx_subgraph_centrality(G, nodes, k=100)
    cl = nx.clustering(G)
    lac = {}
    for n in nodes:
        neigh = list(G.neighbors(n))
        lac[n] = float(np.mean([deg[m] for m in neigh])) if neigh else 0.0
    nc = {n: deg[n] + sum((deg[m] for m in G.neighbors(n))) for n in nodes}
    core = nx.core_number(G)
    df = pd.DataFrame({'protein': nodes, 'DC': [dc[n] for n in nodes], 'BC': [bc[n] for n in nodes], 'CC': [cc[n] for n in nodes], 'EC': [ec[n] for n in nodes], 'SC': [sc[n] for n in nodes], 'CL': [cl[n] for n in nodes], 'LAC': [lac[n] for n in nodes], 'NC': [nc[n] for n in nodes], 'KC': [core[n] for n in nodes]}).set_index('protein')
    return df


compute_topology = compute_all  # alias used by the main pipeline below


In [4]:
# ============================== validation ==============================
TOP_PCTS = [0.01, 0.05, 0.1, 0.15, 0.2, 0.25]

def top_n_validation(scores: pd.Series, labels: pd.Series):
    """scores, labels indexed by protein, higher score = more likely
    essential. Returns a DataFrame of precision/recall/F1 at each top-N%
    cutoff, restricted to proteins that have a gold-standard label."""
    common = scores.index.intersection(labels.index)
    s = scores.loc[common].sort_values(ascending=False)
    y = labels.loc[s.index]
    n_total = len(s)
    n_true_ess = int(y.sum())
    rows = []
    for pct in TOP_PCTS:
        k = max(1, int(round(pct * n_total)))
        top_labels = y.iloc[:k]
        tp = int(top_labels.sum())
        precision = tp / k
        recall = tp / n_true_ess if n_true_ess else np.nan
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
        rows.append({'top_pct': pct, 'n_selected': k, 'true_positives': tp, 'precision': precision, 'recall': recall, 'f1': f1})
    return pd.DataFrame(rows)

def cross_validated_rf(X: pd.DataFrame, y: pd.Series, n_splits=10, random_state=42):
    """Stratified k-fold CV with a Random Forest classifier. Returns
    (metrics_dict, oof_probabilities, feature_importance_df)."""
    X = X.copy()
    X = X.fillna(X.median(numeric_only=True))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof_proba = np.zeros(len(y))
    importances = np.zeros(X.shape[1])
    for train_idx, test_idx in skf.split(X, y):
        clf = RandomForestClassifier(n_estimators=400, max_depth=None, min_samples_leaf=2, class_weight='balanced', n_jobs=-1, random_state=random_state)
        clf.fit(X.iloc[train_idx], y.iloc[train_idx])
        oof_proba[test_idx] = clf.predict_proba(X.iloc[test_idx])[:, 1]
        importances += clf.feature_importances_ / n_splits
    oof_pred = (oof_proba >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, oof_pred).ravel()
    metrics = {'n_samples': len(y), 'n_essential': int(y.sum()), 'n_nonessential': int((y == 0).sum()), 'accuracy': (tp + tn) / (tp + tn + fp + fn), 'sensitivity_recall': tp / (tp + fn) if tp + fn else np.nan, 'specificity': tn / (tn + fp) if tn + fp else np.nan, 'precision_ppv': tp / (tp + fp) if tp + fp else np.nan, 'npv': tn / (tn + fn) if tn + fn else np.nan, 'f1': 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan, 'mcc': matthews_corrcoef(y, oof_pred), 'roc_auc': roc_auc_score(y, oof_proba), 'pr_auc': average_precision_score(y, oof_proba)}
    fi = pd.Series(importances, index=X.columns).sort_values(ascending=False)
    return (metrics, oof_proba, fi)


In [6]:
# ============================== main pipeline ==============================
matplotlib.use('Agg')
OUT_DIR = '/mnt/user-data/outputs'
UPLOAD_DIR = '/content' # Explicitly setting UPLOAD_DIR for the main pipeline
os.makedirs(OUT_DIR, exist_ok=True)
DATASETS = ['YDIP', 'YMIPS', 'YHQ', 'YMBD']
CENTRALITY_COLS = ['DC', 'BC', 'CC', 'EC', 'SC', 'CL', 'LAC', 'NC', 'KC']
gold = load_gold_standard()
gold_series_full = pd.Series(gold)
pc_features = load_physico_chemical()
summary_rf = []
summary_topn_best = []
all_topn_tables = {}
all_rf_metrics = {}
all_feature_importance = {}
for name in DATASETS:
    print(f'\n=== {name} ===')
    G = build_graph(name)
    print(f'nodes={G.number_of_nodes()} edges={G.number_of_edges()}')
    topo = compute_topology(G)
    go_scores = load_go_scores(name)
    topo['GO'] = topo.index.map(go_scores).astype(float)
    feats = topo.join(pc_features, how='left')
    if name == 'YDIP':
        ortho = load_cog_orthology('YDIP')
        feats['ORTHO'] = feats.index.map(ortho).fillna(0)
    feats.to_csv(f'{OUT_DIR}/{name}_features.csv')
    score_cols = CENTRALITY_COLS + ['GO'] + (['ORTHO'] if 'ORTHO' in feats else [])
    topn_tables = {}
    for col in score_cols:
        s = feats[col].dropna()
        table = top_n_validation(s, gold_series_full)
        topn_tables[col] = table
    all_topn_tables[name] = topn_tables
    prec10 = {col: t.loc[t.top_pct == 0.1, 'precision'].values[0] for col, t in topn_tables.items()}
    best_col = max(prec10, key=prec10.get)
    summary_topn_best.append({'dataset': name, 'best_single_score': best_col, 'precision_at_top10pct': prec10[best_col]})
    plt.figure(figsize=(6, 4.5))
    for col in ['DC', 'BC', 'CC', 'EC', 'SC', 'GO']:
        t = topn_tables[col]
        plt.plot([p * 100 for p in t.top_pct], t.precision, marker='o', label=col)
    plt.xlabel('Top N% of ranked proteins')
    plt.ylabel('Precision (fraction truly essential)')
    plt.title(f'{name}: top-N% precision by scoring method')
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/{name}_topN_precision.png', dpi=150)
    plt.close()
    labeled = feats.join(gold_series_full.rename('label'), how='inner')
    X = labeled[score_cols]
    y = labeled['label'].astype(int)
    metrics, oof_proba, fi = cross_validated_rf(X, y)
    metrics['dataset'] = name
    all_rf_metrics[name] = metrics
    all_feature_importance[name] = fi
    summary_rf.append(metrics)
    fi.to_csv(f'{OUT_DIR}/{name}_RF_feature_importance.csv', header=['importance'])
    fpr, tpr, _ = roc_curve(y, oof_proba)
    prec, rec, _ = precision_recall_curve(y, oof_proba)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(fpr, tpr, label=f'AUC={metrics['roc_auc']:.3f}')
    axes[0].plot([0, 1], [0, 1], 'k--', lw=0.8)
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title(f'{name}: ROC (10-fold CV)')
    axes[0].legend()
    axes[1].plot(rec, prec, label=f'AP={metrics['pr_auc']:.3f}')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title(f'{name}: Precision-Recall (10-fold CV)')
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/{name}_RF_ROC_PR.png', dpi=150)
    plt.close()
    print(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()}, indent=2))
rf_summary_df = pd.DataFrame(summary_rf).set_index('dataset')
rf_summary_df.to_csv(f'{OUT_DIR}/RF_crossvalidation_summary.csv')
topn_best_df = pd.DataFrame(summary_topn_best).set_index('dataset')
topn_best_df.to_csv(f'{OUT_DIR}/topN_best_single_score_summary.csv')
plt.figure(figsize=(6, 4))
plt.bar(rf_summary_df.index, rf_summary_df['roc_auc'], color='#4C72B0')
plt.ylim(0, 1)
plt.ylabel('ROC-AUC (10-fold CV)')
plt.title('Random Forest essentiality prediction: ROC-AUC by dataset')
for i, v in enumerate(rf_summary_df['roc_auc']):
    plt.text(i, v + 0.02, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/crossdataset_ROCAUC_comparison.png', dpi=150)
plt.close()
print('\n\n=== FINAL RF CROSS-VALIDATION SUMMARY ===')
print(rf_summary_df.round(3).to_string())
print('\n=== BEST SINGLE TOPOLOGICAL/BIOLOGICAL SCORE (precision@top10%) ===')
print(topn_best_df.to_string())
print('\nDone. Outputs written to', OUT_DIR)


=== YDIP ===
nodes=5093 edges=24743
{
  "n_samples": 4757,
  "n_essential": 1166,
  "n_nonessential": 3591,
  "accuracy": 0.7555,
  "sensitivity_recall": 0.2787,
  "specificity": 0.9103,
  "precision_ppv": 0.5023,
  "npv": 0.7954,
  "f1": 0.3585,
  "mcc": 0.2372,
  "roc_auc": 0.6573,
  "pr_auc": 0.4224,
  "dataset": "YDIP"
}

=== YMIPS ===
nodes=4546 edges=12319
{
  "n_samples": 4209,
  "n_essential": 1015,
  "n_nonessential": 3194,
  "accuracy": 0.7292,
  "sensitivity_recall": 0.2562,
  "specificity": 0.8795,
  "precision_ppv": 0.4031,
  "npv": 0.7882,
  "f1": 0.3133,
  "mcc": 0.1611,
  "roc_auc": 0.6308,
  "pr_auc": 0.3693,
  "dataset": "YMIPS"
}

=== YHQ ===
nodes=4683 edges=22665
{
  "n_samples": 4302,
  "n_essential": 1099,
  "n_nonessential": 3203,
  "accuracy": 0.7824,
  "sensitivity_recall": 0.4013,
  "specificity": 0.9132,
  "precision_ppv": 0.6134,
  "npv": 0.8164,
  "f1": 0.4851,
  "mcc": 0.3676,
  "roc_auc": 0.7057,
  "pr_auc": 0.5494,
  "dataset": "YHQ"
}

=== YMBD ===
no